In [2]:
import re
from pathlib import Path
from collections import defaultdict

SRC_DIR = Path("/Users/nityaarya/Downloads/blackrock-esg-etf-study/Data/Raw Data/Past holdings/Converted_CSV")
OUT_DIR = Path("/Users/nityaarya/Downloads/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings")
YEARS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

OUT_DIR.mkdir(parents=True, exist_ok=True)

pat = re.compile(rb"_soi_(\d{4})\.csv$", re.IGNORECASE)
files_by_year = defaultdict(list)

for p in sorted(SRC_DIR.glob("*.csv")):
    name_b = p.name.encode("utf-8", "surrogatepass")
    m = pat.search(name_b)
    if not m:
        continue
    year = int(m.group(1).decode("ascii"))
    if year in YEARS:
        files_by_year[year].append(p)

def split_first_line(raw: bytes):
    """Return (first_line_including_terminator, remainder, newline_bytes)."""
    idx_crlf = raw.find(b"\r\n")
    idx_lf = raw.find(b"\n")
    if idx_crlf != -1 and (idx_lf == -1 or idx_crlf < idx_lf):
        end = idx_crlf + 2
        return raw[:end], raw[end:], b"\r\n"
    elif idx_lf != -1:
        end = idx_lf + 1
        return raw[:end], raw[end:], b"\n"
    else:
        return raw, b"", b""

def ensure_trailing_newline(buf: bytearray, newline: bytes):
    if not newline:
        return
    if not buf.endswith(newline):
        buf.extend(newline)

for year in YEARS:
    paths = files_by_year.get(year, [])
    if not paths:
        continue

    combined = bytearray()
    header_newline = b"\n"

    for i, p in enumerate(sorted(paths, key=lambda x: x.name.lower())):
        raw = p.read_bytes()
        header, rest, nl = split_first_line(raw)
        if nl:
            header_newline = nl

        if i == 0:
            combined.extend(header)
            combined.extend(rest)
        else:
            if rest:
                ensure_trailing_newline(combined, header_newline)
                combined.extend(rest)

    out_path = OUT_DIR / f"soi_{year}.csv"
    out_path.write_bytes(bytes(combined))
    print(f"Wrote: {out_path}  (from {len(paths)} file(s))")

print("Done.")


Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2017.csv  (from 8 file(s))
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2018.csv  (from 9 file(s))
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2019.csv  (from 10 file(s))
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2020.csv  (from 14 file(s))
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2021.csv  (from 17 file(s))
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2022.csv  (from 18 file(s))
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2023.csv  (from 19 file(s))
Wrote: /Users/nityaarya/Downloads/blackrock-esg-etf-study/Data/Processe